<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/hvac_langchain_deep_agent_colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HVAC RAG + Deep Agent Demo (Colab)

This notebook demonstrates three answer modes over an existing OpenAI-hosted HVAC document vector store:

- **Baseline LLM**: no retrieval
- **RAG**: retrieval-grounded answer using retrieved HVAC documentation
- **Deep Agent + RAG**: a LangChain Deep Agent that can retrieve, compare, and recommend using multi-step tool use

## Setup

1. Run the install cell.
2. Set `OPENAI_API_KEY` and `OPENAI_VECTOR_STORE_ID` in Colab Secrets or environment variables.
3. Run the config cell.
4. Run the app cell and use the widgets.

Expected secret names:

- `OPENAI_API_KEY`
- `OPENAI_VECTOR_STORE_ID`

The notebook is presenter-oriented: it shows the query, retrieved evidence, a concise agent step trace, and the final answer. The core interactive app stays in one code cell for a simple first cut.

In [1]:
# Colab install cell
%pip -q install -U openai langchain langchain-openai deepagents ipywidgets pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.5/104.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [5]:
import json
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_secret(name: str, default: str = "") -> str:
    if os.getenv(name):
        return os.getenv(name)
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return default


OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
OPENAI_VECTOR_STORE_ID = get_secret("OPENAI_VECTOR_STORE_ID", "vs_your_vector_store_id")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4.1-mini")
AGENT_MODEL = os.getenv("AGENT_MODEL", "openai:gpt-5.4")
DEFAULT_TOP_K = int(os.getenv("TOP_K", "4"))

if not OPENAI_API_KEY:
    raise ValueError("Missing OPENAI_API_KEY. Set it in Colab Secrets or environment variables.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print(json.dumps({
    "CHAT_MODEL": CHAT_MODEL,
    "AGENT_MODEL": AGENT_MODEL,
    "OPENAI_VECTOR_STORE_ID": OPENAI_VECTOR_STORE_ID,
    "DEFAULT_TOP_K": DEFAULT_TOP_K,
}, indent=2))

if OPENAI_VECTOR_STORE_ID == "vs_your_vector_store_id":
    print("Update OPENAI_VECTOR_STORE_ID before running retrieval-backed modes.")

{
  "CHAT_MODEL": "gpt-4.1-mini",
  "AGENT_MODEL": "openai:gpt-5.4",
  "OPENAI_VECTOR_STORE_ID": "vs_691b4ac63a2481918ceef99207f869d6",
  "DEFAULT_TOP_K": 4
}


In [6]:
# Core demo app cell
import json
import textwrap
from datetime import datetime

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display
from deepagents import create_deep_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
baseline_llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
rag_llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

TRACE_LOG = []


def add_trace(message: str) -> None:
    timestamp = datetime.now().strftime("%H:%M:%S")
    TRACE_LOG.append(f"{timestamp} | {message}")


def message_to_text(message) -> str:
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                if block.get("type") == "text":
                    parts.append(block.get("text", ""))
                elif "text" in block:
                    parts.append(str(block["text"]))
        return "\n".join(part for part in parts if part).strip()
    return str(content)


def to_plain_dict(obj):
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "__dict__"):
        return obj.__dict__
    return {}


def build_user_query(task_type: str, free_form: str, product_a: str, product_b: str, constraints: str, focus_areas: str) -> str:
    free_form = free_form.strip()
    if free_form:
        return free_form

    if task_type == "comparison":
        if not product_a or not product_b:
            raise ValueError("Comparison mode requires Product A and Product B, or a free-form query.")
        areas = focus_areas.strip() or "controls, monitoring, integration support, and notable feature differences"
        extra = f" Constraints: {constraints.strip()}." if constraints.strip() else ""
        return f"Compare {product_a} and {product_b} for {areas}.{extra} Use retrieved HVAC docs and cite supporting evidence."

    if task_type == "recommendation":
        if not constraints.strip():
            raise ValueError("Recommendation mode requires scenario constraints, or a free-form query.")
        return f"Recommend an HVAC product or product family for this scenario: {constraints.strip()} Use retrieved HVAC documentation, explain tradeoffs, and cite evidence."

    if constraints.strip():
        return f"{constraints.strip()}"
    raise ValueError("Enter a free-form question or provide enough structured inputs for the selected task type.")


def build_filter(product_family: str):
    product_family = product_family.strip()
    if not product_family:
        return None
    return {"type": "eq", "key": "product_family", "value": product_family}


def normalize_search_results(search_response):
    payload = to_plain_dict(search_response)
    raw_items = payload.get("data") or getattr(search_response, "data", []) or []
    normalized = []
    for item in raw_items:
        row = to_plain_dict(item)
        content_blocks = row.get("content") or []
        text_parts = []
        for block in content_blocks:
            block_dict = to_plain_dict(block)
            if block_dict.get("type") == "text" and block_dict.get("text"):
                text_parts.append(block_dict["text"])
        snippet = "\n\n".join(text_parts).strip()
        normalized.append({
            "file_id": row.get("file_id", ""),
            "filename": row.get("filename") or row.get("file_name") or "unknown",
            "score": round(float(row.get("score", 0.0)), 4),
            "attributes": row.get("attributes") or {},
            "snippet": snippet,
        })
    return normalized


def search_hvac_docs(query: str, top_k: int = DEFAULT_TOP_K, product_family: str = ""):
    add_trace(f"Retrieval invoked for query: {query}")
    if OPENAI_VECTOR_STORE_ID == "vs_your_vector_store_id":
        warning = "OPENAI_VECTOR_STORE_ID is still a placeholder, so retrieval-backed modes cannot run yet."
        add_trace(warning)
        return {"query": query, "results": [], "warning": warning, "evidence_text": ""}

    kwargs = {
        "vector_store_id": OPENAI_VECTOR_STORE_ID,
        "query": query,
        "max_num_results": int(top_k),
        "rewrite_query": True,
    }
    metadata_filter = build_filter(product_family)
    if metadata_filter:
        kwargs["filters"] = metadata_filter
        add_trace(f"Applied metadata filter product_family={product_family}")

    try:
        response = openai_client.vector_stores.search(**kwargs)
        results = normalize_search_results(response)
        add_trace(f"Retrieved {len(results)} chunk(s) from vector store {OPENAI_VECTOR_STORE_ID}")
    except Exception as exc:
        warning = f"Retrieval failed: {exc}"
        add_trace(warning)
        return {"query": query, "results": [], "warning": warning, "evidence_text": ""}

    evidence_lines = []
    for idx, item in enumerate(results, start=1):
        snippet = item["snippet"] or "[No text snippet returned]"
        evidence_lines.append(
            f"[{idx}] {item['filename']} | score={item['score']}\n{snippet}"
        )

    warning = ""
    if not results:
        warning = "No matching chunks were retrieved. The answer should explicitly say evidence was insufficient."
        add_trace(warning)

    return {
        "query": query,
        "results": results,
        "warning": warning,
        "evidence_text": "\n\n".join(evidence_lines),
    }


baseline_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an HVAC product expert helping with a demo. Answer directly, but do not pretend to have access to internal documents or unseen product manuals.",
    ),
    ("human", "Question: {query}"),
])

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an HVAC product expert. Answer only from the retrieved evidence. If the evidence is incomplete, say so clearly. Cite filenames in square brackets such as [manual.pdf].",
    ),
    (
        "human",
        "Question:\n{query}\n\nRetrieved evidence:\n{evidence_text}\n\nReturn a grounded answer with a short evidence sufficiency note.",
    ),
])

agent_system_prompt = """
You are an HVAC documentation analyst.

Use the `search_hvac_docs_tool` whenever a task depends on product facts, comparisons, protocols, options, controls, monitoring, or recommendations.

Goals:
- Ground answers in retrieved HVAC documentation.
- For comparison tasks, compare both products using retrieved evidence.
- For recommendation tasks, explain tradeoffs and caveats.
- Cite filenames in square brackets.
- If retrieval is weak or incomplete, say so plainly.

Do not reveal chain-of-thought. Provide the final answer only.
""".strip()


def run_baseline(query: str):
    add_trace("Baseline LLM mode started")
    response = baseline_llm.invoke(baseline_prompt.format_messages(query=query))
    add_trace("Baseline LLM mode finished")
    return {"answer": message_to_text(response), "evidence": [], "trace": TRACE_LOG.copy(), "warning": ""}


def run_rag(query: str, top_k: int, product_family: str):
    add_trace("RAG mode started")
    retrieval = search_hvac_docs(query, top_k=top_k, product_family=product_family)
    if retrieval["results"]:
        response = rag_llm.invoke(rag_prompt.format_messages(query=query, evidence_text=retrieval["evidence_text"]))
        answer = message_to_text(response)
    else:
        answer = "I could not ground an answer because no supporting HVAC snippets were retrieved from the configured vector store."
    add_trace("RAG mode finished")
    return {
        "answer": answer,
        "evidence": retrieval["results"],
        "trace": TRACE_LOG.copy(),
        "warning": retrieval["warning"],
    }


def run_deep_agent(query: str, top_k: int, product_family: str):
    tool_evidence = []

    def search_hvac_docs_tool(tool_query: str) -> str:
        """Search the OpenAI-hosted HVAC vector store and return the most relevant document chunks as JSON."""
        retrieval = search_hvac_docs(tool_query, top_k=top_k, product_family=product_family)
        tool_evidence.extend(retrieval["results"])
        return json.dumps({
            "query": tool_query,
            "warning": retrieval["warning"],
            "results": retrieval["results"],
        }, indent=2)

    add_trace("Deep Agent mode started")
    agent = create_deep_agent(
        model=AGENT_MODEL,
        tools=[search_hvac_docs_tool],
        system_prompt=agent_system_prompt,
    )
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    answer = message_to_text(result["messages"][-1])
    add_trace("Deep Agent mode finished")

    unique_evidence = []
    seen = set()
    for item in tool_evidence:
        key = (item.get("file_id"), item.get("snippet"))
        if key in seen:
            continue
        seen.add(key)
        unique_evidence.append(item)

    warning = ""
    if not unique_evidence and OPENAI_VECTOR_STORE_ID == "vs_your_vector_store_id":
        warning = "OPENAI_VECTOR_STORE_ID is still a placeholder, so the agent could not retrieve evidence."

    return {
        "answer": answer,
        "evidence": unique_evidence,
        "trace": TRACE_LOG.copy(),
        "warning": warning,
    }


def render_evidence(evidence):
    if not evidence:
        display(Markdown("### Retrieved Evidence\n_No retrieved evidence for this mode._"))
        return

    display(Markdown("### Retrieved Evidence"))
    evidence_df = pd.DataFrame([
        {
            "filename": item["filename"],
            "score": item["score"],
            "file_id": item["file_id"],
            "snippet": textwrap.shorten(item["snippet"].replace("\n", " "), width=180, placeholder=" ..."),
        }
        for item in evidence
    ])
    display(evidence_df)

    details = []
    for idx, item in enumerate(evidence, start=1):
        snippet = item["snippet"] or "[No snippet returned]"
        details.append(
            f"**{idx}. {item['filename']}**  \nscore={item['score']} | file_id={item['file_id']}\n\n> {snippet.replace(chr(10), chr(10) + '> ')}"
        )
    display(Markdown("\n\n".join(details)))


def render_trace(trace):
    if not trace:
        display(Markdown("### Agent Steps\n_No trace captured._"))
        return
    bullet_lines = "\n".join(f"- {line}" for line in trace)
    display(Markdown(f"### Agent Steps\n{bullet_lines}"))


def render_result(mode_label: str, query: str, result: dict):
    display(Markdown(f"## {mode_label}"))
    display(Markdown(f"**User Query**\n\n{query}"))
    if result.get("warning"):
        display(Markdown(f"> {result['warning']}"))
    render_evidence(result.get("evidence", []))
    render_trace(result.get("trace", []))
    display(Markdown(f"### Final Answer\n{result.get('answer', '').strip()}"))


mode_widget = widgets.Dropdown(
    options=[("All modes", "all"), ("Baseline only", "baseline"), ("RAG only", "rag"), ("Deep Agent only", "deep_agent")],
    value="all",
    description="Mode:",
    layout=widgets.Layout(width="320px"),
)

task_widget = widgets.Dropdown(
    options=[("Grounded Q&A", "qa"), ("Product comparison", "comparison"), ("Recommendation", "recommendation")],
    value="qa",
    description="Task:",
    layout=widgets.Layout(width="320px"),
)

query_widget = widgets.Textarea(
    value="Which rooftop unit supports BACnet/IP?",
    description="Query:",
    layout=widgets.Layout(width="100%", height="100px"),
)

product_a_widget = widgets.Text(value="", description="Product A:", layout=widgets.Layout(width="48%"))
product_b_widget = widgets.Text(value="", description="Product B:", layout=widgets.Layout(width="48%"))
focus_widget = widgets.Text(value="economizer support and remote monitoring", description="Focus:", layout=widgets.Layout(width="100%"))
constraints_widget = widgets.Textarea(value="", description="Constraints:", layout=widgets.Layout(width="100%", height="80px"))
product_family_widget = widgets.Text(value="", description="Filter:", layout=widgets.Layout(width="320px"))
top_k_widget = widgets.IntSlider(value=DEFAULT_TOP_K, min=1, max=8, step=1, description="Top K:", continuous_update=False)
run_button = widgets.Button(description="Run Demo", button_style="primary")
output = widgets.Output()


def on_run_clicked(_):
    with output:
        clear_output(wait=True)
        TRACE_LOG.clear()
        try:
            query = build_user_query(
                task_type=task_widget.value,
                free_form=query_widget.value,
                product_a=product_a_widget.value,
                product_b=product_b_widget.value,
                constraints=constraints_widget.value,
                focus_areas=focus_widget.value,
            )
            top_k = int(top_k_widget.value)
            product_family = product_family_widget.value
            display(Markdown("# HVAC Demo Output"))
            display(Markdown(f"**Selected Mode**: `{mode_widget.value}`  \n**Task Type**: `{task_widget.value}`"))

            if mode_widget.value in ("all", "baseline"):
                TRACE_LOG.clear()
                render_result("Baseline LLM", query, run_baseline(query))

            if mode_widget.value in ("all", "rag"):
                TRACE_LOG.clear()
                render_result("RAG", query, run_rag(query, top_k=top_k, product_family=product_family))

            if mode_widget.value in ("all", "deep_agent"):
                TRACE_LOG.clear()
                render_result("Deep Agent + RAG", query, run_deep_agent(query, top_k=top_k, product_family=product_family))
        except Exception as exc:
            display(Markdown(f"## Error\n`{exc}`"))


run_button.on_click(on_run_clicked)

display(Markdown("# HVAC LangChain Deep Agent Demo"))
display(Markdown("Use a free-form query or the structured fields. For comparison or recommendation demos, you can either type the full prompt in the query box or leave it empty and use the structured inputs."))
display(widgets.VBox([
    widgets.HBox([mode_widget, task_widget]),
    query_widget,
    widgets.HBox([product_a_widget, product_b_widget]),
    focus_widget,
    constraints_widget,
    widgets.HBox([product_family_widget, top_k_widget]),
    run_button,
    output,
]))

# HVAC LangChain Deep Agent Demo

Use a free-form query or the structured fields. For comparison or recommendation demos, you can either type the full prompt in the query box or leave it empty and use the structured inputs.

In [4]:
SAMPLE_QUERIES = [
    "Which rooftop unit supports BACnet/IP?",
    "Compare Product X and Product Y for economizer support and remote monitoring.",
    "Recommend an HVAC option for a small school in a humid climate with remote monitoring needs.",
]

for idx, query in enumerate(SAMPLE_QUERIES, start=1):
    print(f"{idx}. {query}")

1. Which rooftop unit supports BACnet/IP?
2. Compare Product X and Product Y for economizer support and remote monitoring.
3. Recommend an HVAC option for a small school in a humid climate with remote monitoring needs.


## Notes For Extension

- Add richer metadata filters by extending `build_filter`.
- Port the retrieval helpers and rendering logic into Streamlit later with minimal changes.
- If you want a stricter grounded mode, tighten the RAG prompt so it refuses to answer unless retrieval returns evidence.
- If you want richer Deep Agent traces, add LangSmith or LangGraph tracing later without exposing chain-of-thought.